# "But wait... there's more"

## A More Visible Agent Loop

The Digital Twin contained an Agent Loop. But it was behind-the-scenes, running every time the user asked a message. Using its tools and then replying. It didn't feel very... loopy.

### Adding 2 more ingredients to make it more real

Let's make an Agent Loop with some familiar features borrowed from Claude Code:

1. A Terminal UI (TUI)
2. A Checklist tool to cause and track multiple tool calls


In [ ]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
load_dotenv(override=True)

In [ ]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [ ]:
# Free-first bootstrap: makes bare `OpenAI()` work without an OpenAI key,
# and provides a universal `llm_call(messages, ...)` helper with multi-model/provider fallback.
#
# Self-contained: runs even if Cell 1 imports haven't executed yet on this kernel.

import os
import json
from dotenv import load_dotenv
from openai import OpenAI as _LLMOpenAI

load_dotenv(override=True)

_gr_key   = os.getenv("GROQ_API_KEY")
_gm_key   = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
_or_key   = os.getenv("OPENROUTER_API_KEY")
_oa_key   = os.getenv("OPENAI_API_KEY")

# --- FREE-first env override (backwards compat: existing bare OpenAI() works) ---
if _gr_key:
    _which = "Groq (FREE tier - fastest inference)"
    _key, _base = _gr_key, "https://api.groq.com/openai/v1"
elif _gm_key:
    _which = "Gemini (FREE tier - high quality)"
    _key, _base = _gm_key, "https://generativelanguage.googleapis.com/v1beta/openai/"
elif _or_key:
    _which = "OpenRouter (FREE models gateway)"
    _key, _base = _or_key, "https://openrouter.ai/api/v1"
elif _oa_key:
    _which = "OpenAI (paid tier)"
    _key, _base = _oa_key, None
else:
    _which = "Ollama (local - 100% FREE)"
    _key, _base = "ollama", "http://localhost:11434/v1"

os.environ["OPENAI_API_KEY"] = _key
if _base:
    os.environ["OPENAI_BASE_URL"] = _base
elif "OPENAI_BASE_URL" in os.environ:
    del os.environ["OPENAI_BASE_URL"]

# --- Build universal `llm_call()` helper with cross-model/provider fallback ---

_MODEL_CHAINS = []

if _gr_key:
    _client = _LLMOpenAI(api_key=_gr_key, base_url="https://api.groq.com/openai/v1")
    _MODEL_CHAINS.append((_client, [
        ("llama3-70b-8192",       "Groq FREE Llama 3 70B"),
        ("gemma2-9b-it",          "Groq FREE Gemma 2 9B"),
        ("llama-3.1-8b-instant",  "Groq FREE Llama 3.1 8B"),
        ("mixtral-8x7b-32768",    "Groq FREE Mixtral 8x7B"),
    ]))

if _gm_key:
    _client = _LLMOpenAI(api_key=_gm_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
    _MODEL_CHAINS.append((_client, [
        ("gemini-2.5-flash",      "Gemini FREE 2.5 Flash"),
        ("gemini-2.0-flash",      "Gemini FREE 2.0 Flash"),
    ]))

if _or_key:
    _client = _LLMOpenAI(api_key=_or_key, base_url="https://openrouter.ai/api/v1")
    _MODEL_CHAINS.append((_client, [
        ("nvidia/nemotron-3-ultra-550b-a55b:free", "OR FREE Nemotron 3 Ultra"),
        ("poolside/laguna-s-2.1:free",             "OR FREE Laguna S 2.1"),
        ("deepseek/deepseek-r1:free",              "OR FREE DeepSeek R1"),
        ("meta-llama/llama-3.3-70b-instruct:free", "OR FREE Llama 3.3 70B"),
        ("openrouter/free",                        "OR FREE auto-router"),
    ]))

_OLLAMA = _LLMOpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
_MODEL_CHAINS.append((_OLLAMA, [
    ("llama3.2",    "Ollama llama3.2"),
    ("qwen2.5:3b",  "Ollama qwen2.5 3B"),
]))

if _oa_key:
    _client = _LLMOpenAI(api_key=_oa_key)
    _MODEL_CHAINS.append((_client, [
        ("gpt-5.4-mini",  "OpenAI PAID gpt-5.4-mini"),
        ("gpt-4.1-mini",  "OpenAI PAID gpt-4.1-mini"),
    ]))

_llm_used = None

def llm_call(messages, *, model=None, temperature=None, max_tokens=None,
             tools=None, tool_choice=None, want_response_object=False, verbose=False):
    """Universal FREE-first LLM call with cross-model/provider fallback.

    Returns answer string by default; pass want_response_object=True to get
    raw ChatCompletion (for inspecting tool_calls in an agent loop).
    """
    global _llm_used

    if model is not None:
        try_chain = [(_MODEL_CHAINS[0][0], [(model, "Override: {}".format(model))])]
    else:
        try_chain = [(c, list(ms)) for c, ms in _MODEL_CHAINS]

    last_err = None
    for client, models in try_chain:
        for model_name, label in models:
            kwargs = dict(model=model_name, messages=messages)
            if temperature is not None: kwargs["temperature"] = temperature
            if max_tokens is not None:    kwargs["max_tokens"] = max_tokens
            if tools is not None:         kwargs["tools"] = tools
            if tool_choice is not None:   kwargs["tool_choice"] = tool_choice
            try:
                if verbose:
                    show("[LLM] {} ({})".format(label, model_name))
                resp = client.chat.completions.create(**kwargs)
                _llm_used = (model_name, label)
                if want_response_object:
                    return resp
                msg = resp.choices[0].message
                return msg.content if msg.content is not None else msg
            except Exception as e:
                last_err = e
                if verbose:
                    show("[LLM] failed: {}".format(e))
                continue

    raise RuntimeError(
        "llm_call failed on ALL providers/models. Last error: {}".format(last_err)
    )


# Backwards compat: keep `openai = OpenAI()` (now routed via FREE env override)
openai = OpenAI()

show("BOOTSTRAP OK - Default bare OpenAI() provider: {}".format(_which))
if _base:
    show("BOOTSTRAP OK - Endpoint override: {}".format(_base))
show("BOOTSTRAP OK - llm_call() ready with {} fallback chains (FREE-first)".format(len(_MODEL_CHAINS)))


In [ ]:
# Some lists!

checklist = []
completed = []

In [ ]:
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

In [ ]:
get_checklist_report()

In [ ]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [ ]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [ ]:
checklist, completed = [], []

create_checklist(["Buy groceries", "Finish week 1", "Eat banana"])

In [ ]:
mark_complete(1, "bought")

In [ ]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [ ]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [ ]:
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]

In [ ]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [ ]:
def loop(messages):
    # FREE-first LLM call via the helper (Cell 3).
    # want_response_object=True so we get the raw ChatCompletion object with
    # access to .choices[0].finish_reason and .choices[0].message.tool_calls.
    response = llm_call(messages, tools=tools, want_response_object=True)

    while response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        tool_calls  = message_obj.tool_calls

        # handle_tool_calls() returns list of complete message dicts:
        #   [ {"role": "tool", "content": json.dumps(result), "tool_call_id": id}, ... ]
        # So we use messages.extend(results) — exact same contract as original code.
        results = handle_tool_calls(tool_calls)
        messages.append(message_obj)
        messages.extend(results)

        # Next agent turn — LLM reads tool results and either gives an answer
        # or requests another tool call.
        response = llm_call(messages, tools=tools, want_response_object=True)

    show(response.choices[0].message.content)


In [ ]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [ ]:
checklist, completed = [], []
loop(messages)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>